# omicsTL — Viral vs Bacterial Classification

This notebook demonstrates how to use the **omicsTL transfer-learning pipeline** to classify
samples as viral or bacterial, and how to load pre-trained models for prediction in an
external application (e.g. BacterAI).

---

## Environment Setup

The package has tight dependencies (Python 3.12, R ≥ 4.2.0, and several bioinformatics
libraries). **Docker is the recommended and easiest path.**

### Option 1 — Docker (recommended)

Clone the repository and build the container:

```bash
git clone <repo-url>
cd timed-hpc
docker build . -t omicstl
docker run -it omicstl /bin/bash
```

To load pre-trained model files (`.pth` / `.pkl`) from your local machine, mount the
directory containing them into the container:

```bash
docker run -it -v /path/to/your/models:/models omicstl /bin/bash
# inside the container the files are at /models/mlp_model.pth, /models/rf_model.pkl
```

### Option 2 — VS Code Dev Container

If you are using Visual Studio Code, open the repository and press
`Ctrl+Shift+P` / `Cmd+Shift+P`, then select **Dev Containers: Reopen in Container**.

### Option 3 — Local setup (not recommended)

If Docker is not available, install the following manually before proceeding:

```
Python == 3.12  (setuptools, wheel)
R >= 4.2.0      (BiocManager == 3.20)
```

Use a Python virtual environment and set `R_LIBS_USER` to a custom R library path to
avoid conflicts with existing packages.

---

## Installation

Inside the container (or local environment), install the package from the repo root:

```bash
# Standard install
pip install .

# Development / editable install
pip install -e .
```

---

# Setup packages

In [1]:
import omicstl
import pandas as pd
import numpy as np
from pathlib import Path

# Bring in Data


In [2]:
from pathlib import Path

repo_root = Path("/workspaces/timed-hpc")
data_dir = repo_root / "viral_use_case" / "data"
out_dir = repo_root / "viral_use_case" / "model_outputs"
out_dir.mkdir(parents=True, exist_ok=True)

source_path = data_dir / "source_dset.csv"
target_transfer_path = data_dir / "target_transfer.csv"
target_validation_path = data_dir / "target_validation.csv"

print("Using files:")
print(source_path)
print(target_transfer_path)
print(target_validation_path)

Using files:
/workspaces/timed-hpc/viral_use_case/data/source_dset.csv
/workspaces/timed-hpc/viral_use_case/data/target_transfer.csv
/workspaces/timed-hpc/viral_use_case/data/target_validation.csv


In [3]:
# Read data
# =========================================
base_source_data = pd.read_csv(source_path).set_index("SampleID")
base_target_transfer_data = pd.read_csv(target_transfer_path).set_index("SampleID")
base_target_validation_data = pd.read_csv(target_validation_path).set_index("SampleID")

for name, df in {
    "source": base_source_data,
    "target_transfer": base_target_transfer_data,
    "target_validation": base_target_validation_data,
}.items():
    if "Resp" not in df.columns:
        raise ValueError(f"{name} is missing 'Resp'")
    df["Resp"] = df["Resp"].apply(lambda x: 2 if x == "viral" else 1)

print("\nShapes:")
print("source:", base_source_data.shape)
print("target_transfer:", base_target_transfer_data.shape)
print("target_validation:", base_target_validation_data.shape)

print("\nResponse counts:")
print(base_source_data["Resp"].value_counts(dropna=False))
print(base_target_transfer_data["Resp"].value_counts(dropna=False))
print(base_target_validation_data["Resp"].value_counts(dropna=False))


Shapes:
source: (358, 476)
target_transfer: (192, 476)
target_validation: (48, 476)

Response counts:
Resp
1    179
2    179
Name: count, dtype: int64
Resp
2    147
1     45
Name: count, dtype: int64
Resp
2    33
1    15
Name: count, dtype: int64


# Setup Data Container

In [4]:
# =========================================
#  Dataset container
# =========================================
from omicstl.simulation_utils.data_utils import DatasetContainer
all_datasets = DatasetContainer(
    source_data=base_source_data,
    target_data=base_target_transfer_data,
    target_test_data=[base_target_validation_data],
)
all_datasets.set_response_column("Resp")

source_input = all_datasets.source_data.drop(columns=["Resp"]).copy()
target_input = all_datasets.target_test_data[0].drop(columns=["Resp"]).copy()
target_truth = all_datasets.target_test_data[0]["Resp"].copy()

feature_names = source_input.columns.tolist()

print("\nFeature matrix shapes:")
print("source_input:", source_input.shape)
print("target_input:", target_input.shape)



Feature matrix shapes:
source_input: (358, 475)
target_input: (48, 475)


# Model Fitting

In [5]:
# 4. Params
# =========================================
param_grid = {
    "dropout": [0.25, 0.5],
    "n_latent_dims": [2],
    "hidden_dim_base": [6],
    "lr": [0.01, 0.001],
    "source_epochs": [1000],
    "target_epochs": [1000],
    "freeze": ["none"],
    "weight_decay": [1e-4, 1e-2],
    "gamma": [1, 2, 3],
}


In [6]:
# =========================================
#  Fit models
# =========================================
import torch
from torch import device
import random
from omicstl.simulation_utils.model_utils import fit_dl_model, fit_rf_model
random.seed(1123)
torch.manual_seed(42)

mlp_out = mlp_model = mlp_model_targetonly = None
vae_out = vae_model = vae_model_targetonly = None
rf_out = rf_model = None

try:
    mlp_out, mlp_model, mlp_model_targetonly = fit_dl_model(
        all_datasets,
        "mult_mlp",
        device("cpu"),
        param_grid,
    )
    print("\nMLP fit complete")
except Exception as e:
    print("\nFailed MLP")
    print(e)

torch.manual_seed(42)
try:
    vae_out, vae_model, vae_model_targetonly = fit_dl_model(
        all_datasets,
        "mult_vae",
        device("cpu"),
        param_grid,
    )
    print("\nVAE fit complete")
except Exception as e:
    print("\nFailed VAE")
    print(e)

random.seed(42)
try:
    rf_out, rf_model = fit_rf_model(all_datasets)
    print("\nRF fit complete")
except Exception as e:
    print("\nFailed RF")
    print(e)

/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)



MLP fit complete


/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)



VAE fit complete

RF fit complete


/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1237: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)


In [8]:
display(mlp_out)
display(vae_out)
display(rf_out)

,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,NaN,NaN,0.9375,0.955224,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.010,0.0100,3.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,NaN,NaN,0.3125,0.000000,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.0001,1.0


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,NaN,NaN,0.3125,0.0,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.01,3.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,NaN,NaN,0.3125,0.0,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.001,0.01,1.0


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall,roc_auc
0,None,None,test_0,rf,pred_source_full,target,NaN,NaN,0.312500,0.000000,0.000000,0.000000,0.000000,0.363636
1,None,None,test_0,rf,pred_source_full_val,target,NaN,NaN,0.312500,0.000000,0.000000,0.000000,0.000000,0.363636
2,None,None,test_0,rf,pred_0_full,target,NaN,NaN,0.458333,0.518519,-0.039639,0.666667,0.424242,0.783838
3,None,None,test_0,rf,pred_0_full_val,target,NaN,NaN,0.541667,0.576923,0.178076,0.789474,0.454545,0.783838
4,None,None,test_0,rf,pred_1_full,target,NaN,NaN,0.479167,0.545455,-0.011276,0.681818,0.454545,0.761616
5,None,None,test_0,rf,pred_1_full_val,target,NaN,NaN,0.458333,0.500000,-0.005744,0.684211,0.393939,0.761616
6,None,None,test_0,rf,pred_2_full,target,NaN,NaN,0.500000,0.555556,0.050965,0.714286,0.454545,0.777778
7,None,None,test_0,rf,pred_2_full_val,target,NaN,NaN,0.500000,0.586207,-0.016870,0.680000,0.515152,0.777778
8,None,None,test_0,rf,pred_3_full,target,NaN,NaN,0.645833,0.690909,0.349552,0.863636,0.575758,0.792929
9,None,None,test_0,rf,pred_3_full_val,target,NaN,NaN,0.583333,0.655172,0.163073,0.760000,0.575758,0.792929


# Saving and Loading Models (BacterAI Integration)

The `TIMEDClassifierMLP` and `TIMEDClassifierRF` wrapper classes provide the interface
BacterAI needs: **save a trained model to disk, load it in a separate session, and call predict**.

| Class | Model type | File format |
|---|---|---|
| `TIMEDClassifierMLP` | MLP or VAE transfer model | `.pth` (PyTorch) |
| `TIMEDClassifierRF`  | Random Forest transfer model | `.pkl` (pickle) |


## Step 1 — Wrap the trained models and save to disk

In [9]:
from omicstl import TIMEDClassifierMLP, TIMEDClassifierRF

mlp_save_path = out_dir / "mlp_model.pth"
rf_save_path  = out_dir / "rf_model.pkl"

# classes: sorted unique response values from training data.
# argmax index 0 maps to classes[0], index 1 maps to classes[1], etc.
# For regression tasks, omit the classes argument entirely.
response_classes = sorted(target_truth.unique())
print(f"Response classes: {response_classes}")

# Wrap and save MLP
if mlp_model is not None:
    mlp_clf = TIMEDClassifierMLP(mlp_model, classes=response_classes)
    mlp_clf.save(str(mlp_save_path))
    print(f"MLP saved to {mlp_save_path}")

# Wrap and save RF
# Classification vs regression is inferred automatically from the model.
if rf_model is not None:
    rf_clf = TIMEDClassifierRF(rf_model)
    rf_clf.save(str(rf_save_path))
    print(f"RF  saved to {rf_save_path}")

Response classes: [np.int64(1), np.int64(2)]
MLP saved to /workspaces/timed-hpc/viral_use_case/model_outputs/mlp_model.pth
RF  saved to /workspaces/timed-hpc/viral_use_case/model_outputs/rf_model.pkl


## Step 2 — Load from disk and predict

This is exactly what BacterAI will do. No training required — just load the file and call `predict`.

In [12]:
# Load MLP classifier from disk
mlp_labels_loaded = None
if mlp_save_path.exists():
    mlp_clf_loaded = TIMEDClassifierMLP.load(str(mlp_save_path))
    mlp_labels_loaded = mlp_clf_loaded.predict(target_input)
else:
    print("MLP model file not found — run the save cell first.")

In [11]:
# Load RF classifier from disk
rf_labels_loaded = None
if rf_save_path.exists():
    rf_clf_loaded = TIMEDClassifierRF.load(str(rf_save_path))
    rf_labels_loaded = rf_clf_loaded.predict(target_input)
else:
    print("RF model file not found — run the save cell first.")

## Step 3 — Verify save/load produces identical predictions

In [13]:
# Full predictions dataframe — predicted vs true label for every sample
pred_df = pd.DataFrame(index=target_input.index)
pred_df["True Label"] = target_truth.values

if mlp_labels_loaded is not None:
    pred_df["MLP Predicted"] = mlp_labels_loaded
    pred_df["MLP Correct"]   = pred_df["MLP Predicted"] == pred_df["True Label"]

if rf_labels_loaded is not None:
    pred_df["RF Predicted"] = rf_labels_loaded
    pred_df["RF Correct"]   = pred_df["RF Predicted"] == pred_df["True Label"]

display(pred_df)

print("\nAccuracy:")
if "MLP Correct" in pred_df:
    print(f"  MLP: {pred_df['MLP Correct'].mean():.1%}")
if "RF Correct" in pred_df:
    print(f"  RF:  {pred_df['RF Correct'].mean():.1%}")

,True Label,MLP Predicted,MLP Correct,RF Predicted,RF Correct
SampleID,,,,,
Uni_MCK_D3_12_Hr_R5,1,1,True,2,False
Uni_CoV2_IT_D1_12_Hr_R3,2,2,True,2,True
Uni_NL63_D3_12_Hr_R3,2,2,True,1,False
Uni_MCK_D3_12_Hr_R2,1,1,True,2,False
Uni_NL63_D3_12_Hr_R5,2,2,True,2,True
Uni_CoV2_IT_D1_12_Hr_R4,2,2,True,2,True
Uni_CoV2_IT_D3_12_Hr_R4,2,2,True,1,False
Uni_CoV2_IT_D2_12_Hr_R2,2,2,True,1,False
Uni_NL63_D2_12_Hr_R5,2,2,True,1,False



Accuracy:
  MLP: 93.8%
  RF:  54.2%


## BacterAI Integration — Minimal Example

This is all the code BacterAI needs to integrate predictions.
The files `mlp_model.pth` and `rf_model.pkl` are the only artefacts that
need to be shared from the timed-hpc pipeline.

In [ ]:
# ---- Code that lives inside BacterAI ----
#
# from omicstl import TIMEDClassifierMLP, TIMEDClassifierRF
# import pandas as pd
#
# # Load once at startup
# mlp_clf = TIMEDClassifierMLP.load("mlp_model.pth")
# rf_clf  = TIMEDClassifierRF.load("rf_model.pkl")
#
# # For every new batch of samples:
# new_data = pd.read_csv("new_samples.csv").set_index("SampleID")
# new_data = new_data[feature_names]   # align columns to training order
#
# mlp_labels = mlp_clf.predict(new_data)   # ["viral", "bacterial", ...]
# rf_labels  = rf_clf.predict(new_data)    # ["viral", "bacterial", ...]

print("See commented code above for the BacterAI integration snippet.")